# One JRC flood raster: strict native-pixel HTML viewer

Paste the path to one GeoTIFF and run the notebook. It opens only that file, draws its native flooded pixels as georeferenced Folium polygons, displays the map, and saves the identical map as a downloadable standalone HTML file. There is no catalogue scan, preview-grid, raster-image overlay, interpolation, or pixel limit.

`COARSE_MAX_SIZE` controls the low-resolution whole-raster scan that locates candidate flood cells; keep `1200` unless very small events are missed. Every displayed polygon is then read from the original TIFF at native resolution. `DETAIL_MAX_SIZE` controls only colour scaling and map framing; it does not alter pixel geometry. `SOURCE_PADDING_PIXELS` adds context around the map extent; use `0` for the tightest zoom or `300-600` for context.

In [ ]:
from pathlib import Path
import sys

from IPython.display import FileLink, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from flood_preview import build_strict_pixel_folium_map, preview_summary, read_flood_preview

raster_path_text = input('Paste the full path to one JRC .tif file: ').strip().strip(chr(34))
RASTER_PATH = Path(raster_path_text).expanduser()
COARSE_MAX_SIZE = 1200
DETAIL_MAX_SIZE = 1800
SOURCE_PADDING_PIXELS = 600
HTML_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'strict_native_pixel_maps'
if not RASTER_PATH.is_file():
    raise FileNotFoundError(f'Raster not found: {RASTER_PATH}')

print(RASTER_PATH)

In [ ]:
preview = read_flood_preview(
    RASTER_PATH,
    coarse_max_size=COARSE_MAX_SIZE,
    detail_max_size=DETAIL_MAX_SIZE,
    threshold_cm=0.0,
    mask_values=(9999,),
    source_padding_pixels=SOURCE_PADDING_PIXELS,
)

strict_map = build_strict_pixel_folium_map(
    preview,
    threshold_cm=0.0,
    mask_values=(9999,),
    max_cells=None,
)

HTML_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HTML_PATH = HTML_OUTPUT_DIR / f'{RASTER_PATH.stem}_strict_native_pixels.html'
strict_map.save(str(HTML_PATH))

display(preview_summary(preview))
display(strict_map)
print(f'Artifact-free HTML map saved to: {HTML_PATH.resolve()}')
display(FileLink(str(HTML_PATH), result_html_prefix='Download the strict native-pixel HTML map: '))